# Experiment A — Manual Prompt Baseline

**Evolutionary Prompt Optimization for LLM-Based Sentiment Analysis**  
**Member 4 — Experimental Design and Baseline Implementation**

---

## Purpose

This notebook implements **Experiment A** of the controlled three-way comparison:

| Experiment | Method | Description |
|---|---|---|
| **A (this notebook)** | Manual prompting | Evaluate the 8 hand-crafted prompts authored by Member 2 |
| B (`08_…`) | Random generation | Evaluate 50 purely LLM-generated prompts with no manual seeds or evolution |
| C (`09_…`) | Genetic Algorithm | Compare A, B, and Member 3's GA across seeds `[42, 7, 123]` |

### Controlled Protocol (identical across A, B, and C)
- **Dataset**: SST-2 (`data/processed/` — 53,871 `opt` / 13,468 `val`)
- **Model**: `Qwen/Qwen2.5-1.5B-Instruct` (revision `989aa798`, greedy decoding, `max_new_tokens=8`)
- **Optimization sample**: 1,000 stratified examples from `data/processed/opt.csv` (`seed=42`, matching Member 3)
- **Validation split**: all **13,468** examples in `data/processed/val.csv`, evaluated **once** after prompt selection
- **Test set**: never loaded or touched (`load_final_test` never called)
- **Primary metric**: Macro F1 (`f1_macro`); secondary: Accuracy, Precision, Recall, Invalid Rate

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import subprocess
import random
import time
import importlib.util
from pathlib import Path

REPO_URL = "https://github.com/sarrazer24/evolutionary-prompt-optimization.git"
REPO_DIR = Path("/kaggle/working/evolutionary-prompt-optimization")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    if not status:
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)
    else:
        print("⚠ Working tree has local changes; skipping git pull.")

os.chdir(REPO_DIR)

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
}

missing_packages = [
    pkg for mod, pkg in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(mod) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
else:
    print("All required packages are already installed.")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
print("Project directory:", Path.cwd())
print("Git HEAD:", commit)
print("Python version:", sys.version)
print("Setup complete.")

## 2. Imports and Reproducibility

In [ ]:
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from datetime import datetime
from IPython.display import display, FileLink

from src.data.sst2_data import load_saved_splits, stratified_sample
from src.llm.config import ModelConfig
from src.llm.model_interface import LLMInterface
from src.prompts.baseline_prompts import get_baseline_prompts
from src.evaluation.evaluation import PredictionCache, evaluate_prompt, save_results

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Global seed: {SEED}")

## 3. GPU Check

In [ ]:
if not torch.cuda.is_available():
    print("⚠  No GPU detected — inference will be very slow on CPU.")
    print("   On Kaggle: Settings → Accelerator → GPU T4 x2.")
else:
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {name}  ({mem:.1f} GB)")

## 4. Experiment Configuration

Matches `experiments/configs/experiment_A_manual.json` and Member 3's exact split/sample settings.

In [ ]:
DATA_DIR       = Path("data/processed")
N_OPT_EXAMPLES = 1_000   # stratified sample from data/processed/opt.csv (seed 42)
OUTPUT_DIR     = Path("experiments/results/experiment_A_manual")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_CONFIG = {
    "experiment": "A",
    "name": "manual_prompt_baseline",
    "description": "Manual prompt baseline — 8 hand-crafted prompts by Member 2",
    "dataset": "nyu-mll/glue:sst2",
    "data_dir": str(DATA_DIR),
    "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
    "inference": {"greedy": True, "max_new_tokens": 8, "batch_size": 32},
    "split_seed": 42,
    "evaluation_seed": SEED,
    "n_opt_examples": N_OPT_EXAMPLES,
    "val_sampling": "full_validation_split",
    "metric_primary": "f1_macro",
    "metric_secondary": ["accuracy", "precision_macro", "recall_macro", "invalid_rate"],
    "test_set_used": False,
    "created_at": datetime.now().isoformat(),
}

print(json.dumps(EXPERIMENT_CONFIG, indent=2))

## 5. Load Canonical Data (`data/processed/`)

In [ ]:
for fname in ("opt.csv", "val.csv", "split_info.json"):
    if not (DATA_DIR / fname).exists():
        raise FileNotFoundError(f"Missing canonical split file: {DATA_DIR / fname}")

splits = load_saved_splits(str(DATA_DIR))
opt_df = splits.opt
val_df = splits.val

assert len(opt_df) == 53871, f"Expected 53871 opt rows in data/processed, got {len(opt_df)}"
assert len(val_df) == 13468, f"Expected 13468 val rows in data/processed, got {len(val_df)}"

print(f"Canonical splits loaded from: {DATA_DIR}")
print(f"Optimisation split : {len(opt_df):>6,} rows  (positive: {opt_df['label'].mean():.1%})")
print(f"Validation split   : {len(val_df):>6,} rows  (positive: {val_df['label'].mean():.1%})")

# Stratified 1,000-example optimization sample (identical to Notebooks 05 & 06)
opt_sample = stratified_sample(opt_df, n=N_OPT_EXAMPLES, seed=SEED)
opt_texts  = opt_sample["text"].tolist()
opt_labels = opt_sample["label"].tolist()

# Validation uses the ENTIRE validation split (13,468 rows), identical to Notebooks 05 & 06
val_texts  = val_df["text"].astype(str).tolist()
val_labels = val_df["label"].astype(int).tolist()
N_VAL_EXAMPLES = len(val_texts)
EXPERIMENT_CONFIG["n_val_examples"] = N_VAL_EXAMPLES

(OUTPUT_DIR / "experiment_config.json").write_text(
    json.dumps(EXPERIMENT_CONFIG, indent=4, ensure_ascii=False), encoding="utf-8"
)

print(f"\nOpt sample : {len(opt_texts)} examples  (counts: {opt_sample['label'].value_counts().sort_index().to_dict()})")
print(f"First opt  : {opt_texts[0]}")
print(f"Val split  : {N_VAL_EXAMPLES:,} examples  (positive: {sum(val_labels)/len(val_labels):.1%})")

## 6. Load the Manual Prompts (Member 2)

All 8 prompts are defined in `src/prompts/baseline_prompts.py`.

In [ ]:
MANUAL_PROMPTS = get_baseline_prompts()   # list of (name, prompt_text)

print(f"Number of manual prompts: {len(MANUAL_PROMPTS)}\n")
for i, (name, text) in enumerate(MANUAL_PROMPTS, 1):
    print(f"[{i}] {name}")
    print(f"    {text}")
    print()

## 7. Load the LLM

In [ ]:
model_config = ModelConfig()   # Qwen2.5-1.5B-Instruct, pinned revision, greedy classification
llm = LLMInterface(model_config)
llm.load_model()
print(f"Model loaded: {llm.model_id}")
print(f"Device      : {llm.device}")
print(f"Inference   : {llm.inference_config}")

## 8. Evaluate Every Manual Prompt on the Optimisation Sample

All 8 prompts are scored on **the exact same 1,000-example sample** (identical to Member 3's GA runs).

In [ ]:
cache_A = PredictionCache()
raw_opt_results = []
opt_results = []

t_opt_start = time.perf_counter()
for idx, (name, prompt) in enumerate(MANUAL_PROMPTS, 1):
    print(f"[{idx}/{len(MANUAL_PROMPTS)}] Evaluating: {name}")
    t0 = time.perf_counter()
    result = evaluate_prompt(
        prompt=prompt,
        examples=opt_texts,
        labels=opt_labels,
        predict_fn=llm.predict_raw_batch,
        model_id=llm.model_id,
        inference_config=llm.inference_config,
        cache=cache_A,
        split_name="opt",
    )
    elapsed = time.perf_counter() - t0
    raw_opt_results.append(result)
    m = result["metrics"]
    opt_results.append({
        "rank": None,
        "prompt_name": name,
        "prompt_text": prompt,
        "macro_f1": round(m["f1_macro"], 6),
        "accuracy": round(m["accuracy"], 6),
        "precision_macro": round(m["precision_macro"], 6),
        "recall_macro": round(m["recall_macro"], 6),
        "n_invalid": m["n_invalid"],
        "invalid_rate": round(m["invalid_rate"], 6),
        "n_examples": m["n_examples"],
        "n_cached": result["n_cached"],
        "n_model_texts": result["n_model_texts"],
        "split": "opt",
        "elapsed_s": round(elapsed, 2),
    })
    print(f"   Macro F1 = {m['f1_macro']:.4f}  |  Accuracy = {m['accuracy']:.4f}  |  Invalid = {m['n_invalid']}  |  {elapsed:.1f}s")

opt_total_elapsed = time.perf_counter() - t_opt_start
opt_results.sort(key=lambda r: r["macro_f1"], reverse=True)
for rank, r in enumerate(opt_results, 1):
    r["rank"] = rank

print(f"\nAll manual prompts evaluated in {opt_total_elapsed:.1f}s.")

## 9. Optimisation-Split Leaderboard

In [ ]:
df_opt = pd.DataFrame(opt_results)[
    ["rank", "prompt_name", "macro_f1", "accuracy", "precision_macro", "recall_macro", "n_invalid", "invalid_rate", "elapsed_s"]
]
display(df_opt.set_index("rank"))

best_name   = opt_results[0]["prompt_name"]
best_prompt = opt_results[0]["prompt_text"]
best_f1_opt = opt_results[0]["macro_f1"]
print(f"\n✔ Best manual prompt on opt split: '{best_name}'  (Macro F1 = {best_f1_opt:.4f})")

## 10. Visualise Optimisation Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names  = [r["prompt_name"] for r in opt_results]
f1s    = [r["macro_f1"]    for r in opt_results]
accs   = [r["accuracy"]    for r in opt_results]
colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(names))]

axes[0].barh(names[::-1], f1s[::-1], color=colors[::-1])
axes[0].set_xlabel("Macro F1")
axes[0].set_title("Experiment A — Manual Prompts\n(opt split, ranked by Macro F1)")
axes[0].set_xlim(max(0, min(f1s) - 0.02), min(1, max(f1s) + 0.02))
for v, y in zip(f1s[::-1], range(len(names))):
    axes[0].text(v + 0.001, y, f"{v:.4f}", va="center", fontsize=8)

axes[1].scatter(f1s, accs, s=80, zorder=3)
for name, x, y in zip(names, f1s, accs):
    axes[1].annotate(name, (x, y), textcoords="offset points", xytext=(5, 3), fontsize=7)
axes[1].set_xlabel("Macro F1")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy vs Macro F1 (opt split)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plot_path = OUTPUT_DIR / "manual_prompts_opt.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {plot_path}")

## 11. Validate the Best Manual Prompt (Full 13,468-row Validation Split)

Validation is run **once** on all 13,468 rows of `data/processed/val.csv`, matching Notebooks 05 and 06.

In [ ]:
print(f"Validating best prompt: '{best_name}' on {N_VAL_EXAMPLES:,} examples...")
print(f"Prompt text: {best_prompt}\n")

t0 = time.perf_counter()
val_result = evaluate_prompt(
    prompt=best_prompt,
    examples=val_texts,
    labels=val_labels,
    predict_fn=llm.predict_raw_batch,
    model_id=llm.model_id,
    inference_config=llm.inference_config,
    split_name="val",
)
elapsed_val = time.perf_counter() - t0

vm = val_result["metrics"]
val_summary = {
    "prompt_name":     best_name,
    "prompt_text":     best_prompt,
    "macro_f1":        round(vm["f1_macro"],       6),
    "accuracy":        round(vm["accuracy"],        6),
    "precision_macro": round(vm["precision_macro"], 6),
    "recall_macro":    round(vm["recall_macro"],    6),
    "n_invalid":       vm["n_invalid"],
    "invalid_rate":    round(vm["invalid_rate"],    6),
    "n_examples":      vm["n_examples"],
    "confusion_matrix": vm["confusion_matrix"],
    "split":           "val",
    "elapsed_s":       round(elapsed_val, 2),
}

print(f"Validation Macro F1 : {vm['f1_macro']:.4f}")
print(f"Validation Accuracy : {vm['accuracy']:.4f}")
print(f"Invalid count/rate  : {vm['n_invalid']} ({vm['invalid_rate']:.4%})")
print(f"Elapsed             : {elapsed_val:.1f}s")

## 12. Save Comparison-Ready Results

In [ ]:
# ── Standard Member 1 evaluation format ───────────────────────────────────────
save_results(raw_opt_results, OUTPUT_DIR / "experiment_A_opt_std.csv", append=False)
save_results(raw_opt_results, OUTPUT_DIR / "experiment_A_opt_std.json", include_predictions=False)

# ── Ranked CSV + JSON leaderboard ─────────────────────────────────────────────
opt_csv_path  = OUTPUT_DIR / "experiment_A_opt_results.csv"
opt_json_path = OUTPUT_DIR / "experiment_A_opt_results.json"
pd.DataFrame(opt_results).to_csv(opt_csv_path, index=False)
opt_json_path.write_text(json.dumps(opt_results, indent=2, ensure_ascii=False), encoding="utf-8")

# ── Validation result (includes y_pred for exact McNemar reuse in Notebook 09) ─
val_csv_path  = OUTPUT_DIR / "experiment_A_val_best.csv"
val_json_path = OUTPUT_DIR / "experiment_A_val_best.json"
save_results(val_result, val_csv_path, append=False)
val_payload = dict(val_summary)
val_payload["model_id"] = llm.model_id
val_payload["inference_config"] = llm.inference_config
val_payload["y_pred"] = val_result["y_pred"]
val_json_path.write_text(json.dumps(val_payload, indent=2, ensure_ascii=False), encoding="utf-8")

# ── Comparison-ready summary (consumed by Notebook 09) ───────────────────────
comparison_payload = {
    "experiment": "A",
    "method": "manual_prompts",
    "data_dir": str(DATA_DIR),
    "n_candidates": len(MANUAL_PROMPTS),
    "n_opt_examples": N_OPT_EXAMPLES,
    "n_val_examples": N_VAL_EXAMPLES,
    "best_prompt_name": best_name,
    "best_prompt_text": best_prompt,
    "best_macro_f1_opt": best_f1_opt,
    "best_accuracy_opt": opt_results[0]["accuracy"],
    "best_macro_f1_val": val_summary["macro_f1"],
    "best_accuracy_val": val_summary["accuracy"],
    "best_precision_val": val_summary["precision_macro"],
    "best_recall_val": val_summary["recall_macro"],
    "best_n_invalid_val": val_summary["n_invalid"],
    "opt_elapsed_s": round(opt_total_elapsed, 2),
    "val_elapsed_s": val_summary["elapsed_s"],
    "llm_stats": dict(llm.stats),
    "all_opt_results": opt_results,
    "model_id": llm.model_id,
    "model_revision": model_config.revision,
    "seed": SEED,
    "created_at": datetime.now().isoformat(),
}

summary_path = OUTPUT_DIR / "experiment_A_summary.json"
summary_path.write_text(json.dumps(comparison_payload, indent=4, ensure_ascii=False, default=str), encoding="utf-8")

ZIP_PATH = "/kaggle/working/experiment_A_manual"
shutil.make_archive(ZIP_PATH, "zip", root_dir=str(OUTPUT_DIR))
print("Saved all Experiment A outputs to:", OUTPUT_DIR)
display(FileLink(ZIP_PATH + ".zip"))

## 13. Final Summary

In [ ]:
print("=" * 60)
print("EXPERIMENT A — MANUAL PROMPT BASELINE")
print("=" * 60)
print(f"Data source             : {DATA_DIR}")
print(f"Prompts evaluated       : {len(MANUAL_PROMPTS)}")
print(f"Opt examples per prompt : {N_OPT_EXAMPLES:,}")
print(f"Val examples (best)     : {N_VAL_EXAMPLES:,}")
print(f"LLM classified texts    : {llm.stats['classified_texts']:,}")
print()
print(f"Best prompt             : {best_name}")
print(f"  Opt  Macro F1         : {best_f1_opt:.4f}")
print(f"  Val  Macro F1         : {val_summary['macro_f1']:.4f}")
print(f"  Val  Accuracy         : {val_summary['accuracy']:.4f}")
print()
print("Output files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")
print()
print("Next step: run Notebook 08 (Experiment B — Random Baseline)")